In [2]:
!pip install transformers

In [3]:
!pip install "transformers[torch]"

In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."
...,...,...,...
14727,13863028,Romeo: You are on my ‘People you may know’ lis...,Romeo is trying to get Greta to add him to her...
14728,13828570,Theresa: <file_photo>\r\nTheresa: <file_photo>...,Theresa is at work. She gets free food and fre...
14729,13819050,John: Every day some bad news. Japan will hunt...,Japan is going to hunt whales again. Island an...
14730,13828395,Jennifer: Dear Celia! How are you doing?\r\nJe...,Celia couldn't make it to the afternoon with t...


In [4]:
import re 

def process_dialogue(text):
    text = re.sub(r"\r\n" , " " , text)
    text = re.sub(r"\s+" , " " , text)
    text = re.sub(r"<.*?>" , " " , text)
    text = text.strip().lower()
    return text



    

In [5]:
train_data = train_data.sample(n=4000, random_state= 42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [6]:
train_data["dialogue"] = train_data["dialogue"].apply(process_dialogue)
train_data["summary"] = train_data["summary"].apply(process_dialogue)
val_data["dialogue"] = val_data["dialogue"].apply(process_dialogue)
val_data["summary"] = val_data["summary"].apply(process_dialogue)


In [7]:
train_data["dialogue"]

0       violet: hi! i came across this austin's articl...
1       pat: so does anyone know when the stream is go...
2       jane:   jane: whaddya think? shona: this ur ti...
3       adam: do u have a map of paris? tom: yes, why?...
4       frank: hi, how's the family? mike: great! sam'...
                              ...                        
3995    barry: hello buddy michael: hey barry: do you ...
3996    karen: hey lisa. larissa and me have recently ...
3997    miles: hey, guys, i'm so sorry, but i missed t...
3998    emma: did you finish the book i gave you? liam...
3999    jenna: dudes, were we supposed to read the who...
Name: dialogue, Length: 4000, dtype: str

# Tokenization 

In [8]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [9]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids => add to input as labels
    return inputs

In [10]:
train_set = train_data.apply(tokenize , axis =1).tolist()
val_set = val_data.apply(tokenize , axis =1).tolist()

In [11]:
train_set[1]

{'input_ids': [6234, 10, 78, 405, 1321, 214, 116, 8, 6093, 19, 352, 12, 1837, 58, 16585, 10, 12050, 6, 150, 6, 68, 133, 310, 114, 12, 5, 3, 1050, 2494, 10, 3, 23, 278, 31, 17, 317, 3, 23, 31, 26, 36, 1638, 16, 48, 5, 6234, 10, 3, 63, 58, 3, 1050, 2494, 10, 2492, 66, 8, 1717, 11, 3224, 3640, 7, 656, 140, 1227, 19974, 5, 16585, 10, 78, 25, 31, 60, 78, 7569, 58, 6234, 10, 3, 75, 31, 2157, 55, 3, 7, 52, 7, 120, 58, 3, 1050, 2494, 10, 3, 63, 413, 5, 141, 8, 337, 589, 437, 3, 23, 47, 3, 9, 861, 5, 16585, 10, 2087, 34, 31, 7, 97, 12, 483, 34, 58, 6234, 10, 17945, 55, 428, 34, 3, 9, 653, 55, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

# transformer Summarizer Model

In [12]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [13]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ", device)
model. to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [14]:
training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    # 0 => lr default

)

trainer = Trainer(
    model=model,
    args = training_args,
    train_dataset = train_set,
    eval_dataset = val_set

)

In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.597192,0.379665
2,0.397126,0.359288
3,0.374035,0.353562
4,0.362681,0.350174
5,0.355630,0.348507
6,0.351285,0.348785


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9063246205647787, metrics={'train_runtime': 2246.181, 'train_samples_per_second': 10.685, 'train_steps_per_second': 1.336, 'total_flos': 3248203235328000.0, 'train_loss': 0.9063246205647787, 'epoch': 6.0})

In [15]:
output_dir = "./my_saved_transformer"
# Save the model
model.save_pretrained(output_dir)

# Save the tokenizer (crucial for processing text)
tokenizer.save_pretrained(output_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_saved_transformer\\tokenizer_config.json',
 './my_saved_transformer\\tokenizer.json')